# Logistic Regression

A concise, interview-ready walkthrough of **Logistic Regression** covering theory, math, visualizations, and frequently asked questions.

**Topics Covered:**

1. Why Not Linear Regression for Classification?
2. What is Logistic Regression?
3. The Sigmoid (Logistic) Function
4. Decision Boundary
5. Cost Function -- Log Loss (Binary Cross-Entropy)
6. Gradient Descent for Logistic Regression
7. Maximum Likelihood Estimation (MLE) Connection
8. Assumptions of Logistic Regression
9. Types of Logistic Regression
10. Key sklearn Parameters
11. Odds, Log-Odds, and Interpretation
12. Logistic Regression vs. Linear Regression -- Summary
13. Interview Questions & Answers
14. Quick Reference Summary

> **Scope note:** This notebook covers Session 1 fundamentals. Topics like regularization, multiclass strategies (OvR, OvO), and advanced diagnostics will be added in follow-up sessions.

---
## 1. Import Required Libraries

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.datasets import make_classification

import matplotlib.pyplot as plt

print("Libraries imported successfully!")

---
## 2. Why Not Linear Regression for Classification?

Linear Regression predicts continuous values in the range $(-\infty, +\infty)$. When we try to use it for binary classification (0 or 1), several problems arise:

| Problem | Explanation |
|---|---|
| **Unbounded output** | Predictions can go below 0 or above 1, which makes no sense as a probability. |
| **Sensitive to outliers** | A single extreme data point can shift the decision boundary drastically. |
| **Not a probability** | The output of linear regression is not bounded between [0, 1], so it cannot be interpreted as probability directly. |
| **Violates assumptions** | Errors in classification are not normally distributed, and variance is not constant (heteroscedasticity). |

**Real-world example:** Suppose we predict whether a tumor is malignant (1) or benign (0) based on tumor size. Linear regression might predict values like -0.3 or 1.7 -- neither makes sense as a probability.

**Interview point:** "Linear regression is not suitable for classification because it does not constrain output to [0,1], is sensitive to outliers that distort the decision boundary, and its error distribution assumptions are violated."

In [ ]:
# Demonstration: Linear Regression fails at classification
np.random.seed(42)
X_demo = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 50]).reshape(-1, 1)  # Note: 50 is an outlier
y_demo = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1])

from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression().fit(X_demo, y_demo)
log_reg = LogisticRegression().fit(X_demo, y_demo)

X_plot = np.linspace(-2, 55, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression
axes[0].scatter(X_demo, y_demo, color='black', zorder=5, label='Data')
axes[0].plot(X_plot, lin_reg.predict(X_plot), color='red', label='Linear Regression')
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold 0.5')
axes[0].set_ylim(-0.3, 1.3)
axes[0].set_title('Linear Regression for Classification')
axes[0].set_xlabel('Feature')
axes[0].set_ylabel('Predicted Value')
axes[0].legend()

# Logistic Regression
axes[1].scatter(X_demo, y_demo, color='black', zorder=5, label='Data')
axes[1].plot(X_plot, log_reg.predict_proba(X_plot)[:, 1], color='blue', label='Logistic Regression')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold 0.5')
axes[1].set_ylim(-0.1, 1.1)
axes[1].set_title('Logistic Regression for Classification')
axes[1].set_xlabel('Feature')
axes[1].set_ylabel('Predicted Probability')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Linear Reg prediction for x=50: {lin_reg.predict([[50]])[0]:.2f} (should be close to 1)")
print(f"Linear Reg prediction for x=-2: {lin_reg.predict([[-2]])[0]:.2f} (should be close to 0)")
print("\nNotice how the outlier at x=50 pulls the linear regression line,")
print("shifting the decision boundary and causing misclassifications.")

---
## 3. What is Logistic Regression?

Despite the name, **Logistic Regression is a classification algorithm**, not a regression algorithm. It models the **probability** that a given input belongs to a particular class.

**Core idea:** Apply a sigmoid function to the linear regression output to squeeze it between 0 and 1.

$$z = w_0 + w_1 x_1 + w_2 x_2 + \ldots + w_n x_n = \mathbf{w}^T \mathbf{x}$$

$$\hat{y} = P(y=1 \mid \mathbf{x}) = \sigma(z) = \frac{1}{1 + e^{-z}}$$

Where:
- $z$ is the **log-odds** (linear combination of features)
- $\sigma(z)$ is the **sigmoid function** that maps $z$ to a probability in $(0, 1)$
- $\hat{y}$ is the predicted probability of the positive class

**Interview point:** Logistic regression is a linear model for classification. It is linear in the log-odds space, but nonlinear in the probability space due to the sigmoid transformation.

---
## 4. The Sigmoid (Logistic) Function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Key properties:**

| Property | Value |
|---|---|
| Range | $(0, 1)$ -- always outputs a valid probability |
| At $z = 0$ | $\sigma(0) = 0.5$ |
| As $z \to +\infty$ | $\sigma(z) \to 1$ |
| As $z \to -\infty$ | $\sigma(z) \to 0$ |
| Symmetry | $\sigma(-z) = 1 - \sigma(z)$ |
| Derivative | $\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$ |

The derivative property is important because it makes gradient computation efficient during training.

**Interview point:** The sigmoid derivative $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ means the gradient is largest at $z=0$ (where the model is most uncertain) and vanishes at extremes. This is relevant to the vanishing gradient problem in deep learning.

In [ ]:
def sigmoid(z):
    """Compute sigmoid function."""
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 200)
sig = sigmoid(z)
sig_deriv = sig * (1 - sig)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigmoid function
axes[0].plot(z, sig, color='blue', linewidth=2)
axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='y = 0.5')
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Sigmoid Function')
axes[0].set_xlabel('z')
axes[0].set_ylabel('sigma(z)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sigmoid derivative
axes[1].plot(z, sig_deriv, color='green', linewidth=2)
axes[1].set_title('Sigmoid Derivative')
axes[1].set_xlabel('z')
axes[1].set_ylabel("sigma'(z)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verify key properties
print(f"sigma(0)  = {sigmoid(0):.4f}  (expected: 0.5)")
print(f"sigma(5)  = {sigmoid(5):.4f}  (close to 1)")
print(f"sigma(-5) = {sigmoid(-5):.4f}  (close to 0)")
print(f"sigma(-3) + sigma(3) = {sigmoid(-3) + sigmoid(3):.4f}  (symmetry: should be 1.0)")

---
## 5. Decision Boundary

The decision boundary is the threshold at which the model switches its prediction from one class to another.

**Default rule (threshold = 0.5):**

$$\hat{y} = \begin{cases} 1 & \text{if } \sigma(z) \geq 0.5 \implies z \geq 0 \\ 0 & \text{if } \sigma(z) < 0.5 \implies z < 0 \end{cases}$$

For a 2-feature model: $z = w_0 + w_1 x_1 + w_2 x_2 = 0$ defines a straight line in 2D space.

**Key points:**
- Logistic regression produces a **linear decision boundary** (a line in 2D, a plane in 3D, a hyperplane in higher dimensions).
- The threshold 0.5 is a default. In practice, it can be tuned based on the problem (e.g., use 0.3 for medical diagnosis where missing a positive case is costly).
- Non-linear boundaries can be achieved by adding polynomial features.

**Real-world example:** In fraud detection, you might lower the threshold to 0.3 so that the model flags more transactions as potentially fraudulent, reducing false negatives at the cost of more false positives.

**Interview point:** "The decision boundary in logistic regression is always linear in the original feature space. To get non-linear boundaries, we engineer polynomial or interaction features."

In [ ]:
# Visualize linear decision boundary
from sklearn.datasets import make_classification

X_vis, y_vis = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                    n_informative=2, n_clusters_per_class=1,
                                    random_state=42)

clf = LogisticRegression(random_state=42).fit(X_vis, y_vis)

# Create mesh grid
x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                      np.linspace(y_min, y_max, 300))

Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probability surface
ax = axes[0]
contour = ax.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.8)
ax.scatter(X_vis[y_vis == 0, 0], X_vis[y_vis == 0, 1], c='red', edgecolors='k', label='Class 0')
ax.scatter(X_vis[y_vis == 1, 0], X_vis[y_vis == 1, 1], c='blue', edgecolors='k', label='Class 1')
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
ax.set_title('Decision Boundary (threshold=0.5)')
ax.legend()
plt.colorbar(contour, ax=ax, label='P(y=1)')

# Different thresholds
ax = axes[1]
ax.scatter(X_vis[y_vis == 0, 0], X_vis[y_vis == 0, 1], c='red', edgecolors='k', label='Class 0')
ax.scatter(X_vis[y_vis == 1, 0], X_vis[y_vis == 1, 1], c='blue', edgecolors='k', label='Class 1')
for thresh, color, style in [(0.3, 'green', '--'), (0.5, 'black', '-'), (0.7, 'orange', '--')]:
    ax.contour(xx, yy, Z, levels=[thresh], colors=color, linewidths=2, linestyles=style)
ax.set_title('Effect of Different Thresholds')
ax.legend(['Class 0', 'Class 1', 'Threshold=0.3', 'Threshold=0.5', 'Threshold=0.7'])

plt.tight_layout()
plt.show()

print(f"Model coefficients: w1={clf.coef_[0][0]:.3f}, w2={clf.coef_[0][1]:.3f}")
print(f"Model intercept:    w0={clf.intercept_[0]:.3f}")
print(f"\nDecision boundary equation: {clf.coef_[0][0]:.3f}*x1 + {clf.coef_[0][1]:.3f}*x2 + ({clf.intercept_[0]:.3f}) = 0")

---
## 6. Cost Function -- Log Loss (Binary Cross-Entropy)

We **cannot use MSE** (Mean Squared Error) as the cost function for logistic regression because the sigmoid makes it **non-convex** -- gradient descent would get stuck in local minima.

Instead, we use **Log Loss** (also called Binary Cross-Entropy):

**For a single sample:**

$$\text{Cost}(\hat{y}, y) = \begin{cases} -\log(\hat{y}) & \text{if } y = 1 \\ -\log(1 - \hat{y}) & \text{if } y = 0 \end{cases}$$

**Combined form:**

$$J(\mathbf{w}) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

**Why this works (intuition):**

| True label $y$ | Prediction $\hat{y}$ | Cost | Explanation |
|---|---|---|---|
| 1 | 0.99 | $-\log(0.99) \approx 0.01$ | Correct and confident -- low cost |
| 1 | 0.01 | $-\log(0.01) \approx 4.6$ | Wrong and confident -- very high cost |
| 0 | 0.01 | $-\log(0.99) \approx 0.01$ | Correct and confident -- low cost |
| 0 | 0.99 | $-\log(0.01) \approx 4.6$ | Wrong and confident -- very high cost |

The cost function **heavily penalizes confident wrong predictions**.

**Interview point:** "We use log loss instead of MSE because the sigmoid makes MSE non-convex. Log loss is convex, guaranteeing a global minimum via gradient descent. It also has a probabilistic interpretation -- it is derived from Maximum Likelihood Estimation (MLE)."

In [ ]:
# Visualize the cost function
y_hat = np.linspace(0.001, 0.999, 500)

cost_y1 = -np.log(y_hat)         # Cost when true label y = 1
cost_y0 = -np.log(1 - y_hat)     # Cost when true label y = 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(y_hat, cost_y1, color='blue', linewidth=2)
axes[0].set_title('Cost when y = 1')
axes[0].set_xlabel('Predicted probability (y_hat)')
axes[0].set_ylabel('Cost = -log(y_hat)')
axes[0].grid(True, alpha=0.3)
axes[0].annotate('High cost\n(wrong prediction)', xy=(0.05, 3), fontsize=10, color='red')
axes[0].annotate('Low cost\n(correct prediction)', xy=(0.75, 0.3), fontsize=10, color='green')

axes[1].plot(y_hat, cost_y0, color='red', linewidth=2)
axes[1].set_title('Cost when y = 0')
axes[1].set_xlabel('Predicted probability (y_hat)')
axes[1].set_ylabel('Cost = -log(1 - y_hat)')
axes[1].grid(True, alpha=0.3)
axes[1].annotate('Low cost\n(correct prediction)', xy=(0.02, 0.3), fontsize=10, color='green')
axes[1].annotate('High cost\n(wrong prediction)', xy=(0.7, 3), fontsize=10, color='red')

plt.tight_layout()
plt.show()

---
## 7. Gradient Descent for Logistic Regression

To minimize the log loss, we use gradient descent to update the weights.

**Gradient of the cost function:**

$$\frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \cdot x_j^{(i)}$$

**Update rule:**

$$w_j := w_j - \alpha \cdot \frac{\partial J}{\partial w_j}$$

Where $\alpha$ is the learning rate.

**Important observation:** The gradient formula looks identical to linear regression. The difference is that $\hat{y} = \sigma(\mathbf{w}^T\mathbf{x})$ (sigmoid applied) instead of $\hat{y} = \mathbf{w}^T\mathbf{x}$ (raw linear output).

**Vectorized form:**

$$\nabla J = \frac{1}{m} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$$

$$\mathbf{w} := \mathbf{w} - \alpha \cdot \nabla J$$

**Interview point:** "The gradient update rule for logistic regression has the same form as linear regression, but the prediction function is different. This is not a coincidence -- both belong to the Generalized Linear Model (GLM) family."

---
## 8. Maximum Likelihood Estimation (MLE) Connection

The log loss cost function is derived from **Maximum Likelihood Estimation**.

**Likelihood function** -- probability of observing the training data given the model parameters:

$$L(\mathbf{w}) = \prod_{i=1}^{m} \hat{y}^{(i)^{y^{(i)}}} \cdot (1 - \hat{y}^{(i)})^{(1 - y^{(i)})}$$

Taking the log (for numerical stability and to convert product to sum):

$$\log L(\mathbf{w}) = \sum_{i=1}^{m} \left[ y^{(i)} \log \hat{y}^{(i)} + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

Maximizing the log-likelihood is equivalent to **minimizing the negative log-likelihood**, which is exactly our log loss cost function $J(\mathbf{w})$.

**Interview point:** "Logistic regression parameters are estimated using MLE. The log loss cost function is the negative log-likelihood. Minimizing log loss is mathematically equivalent to finding the most likely parameters given the observed data."

---
## 9. Assumptions of Logistic Regression

| Assumption | Detail |
|---|---|
| **Binary / ordinal / multinomial outcome** | The dependent variable should be categorical (binary for standard logistic regression). |
| **Independence of observations** | Each observation should be independent (no repeated measures without adjustment). |
| **Little or no multicollinearity** | Independent variables should not be highly correlated with each other. Use VIF to check. |
| **Linearity of log-odds** | There must be a linear relationship between the independent variables and the **log-odds** (not the probability itself). |
| **Large sample size** | MLE-based estimation requires a reasonably large sample. Rule of thumb: at least 10 events per predictor variable. |
| **No extreme outliers** | Outliers in the feature space can disproportionately influence the model. |

**What logistic regression does NOT assume:**
- Normal distribution of features (unlike LDA)
- Homoscedasticity (constant variance of errors)
- Linear relationship between features and the dependent variable (it assumes linearity in log-odds, not in probability)

**Interview point:** "Logistic regression assumes linearity in the log-odds space, not in probability space. It does NOT require normally distributed features. The key assumptions to check are: independence of observations, no multicollinearity, and linearity of independent variables with log-odds."

---
## 10. Types of Logistic Regression

| Type | Target Variable | Example |
|---|---|---|
| **Binary** | 2 classes (0/1) | Spam vs. Not Spam, Disease vs. Healthy |
| **Multinomial** | 3+ unordered classes | Predicting animal type (cat, dog, bird) |
| **Ordinal** | 3+ ordered classes | Customer satisfaction (low, medium, high) |

**Binary Logistic Regression** is the most common and is the focus of this session.

For multinomial, sklearn uses **One-vs-Rest (OvR)** or **Softmax (multinomial)** strategies -- these will be covered in a follow-up session.

**Interview point:** "Binary logistic regression uses the sigmoid function. Multinomial logistic regression extends this using the softmax function, which outputs a probability distribution across all classes that sums to 1."

---
## 11. Key sklearn Parameters

| Parameter | Default | Description |
|---|---|---|
| `penalty` | `'l2'` | Regularization type: `'l1'`, `'l2'`, `'elasticnet'`, `None` |
| `C` | `1.0` | Inverse of regularization strength. Smaller C = stronger regularization. |
| `solver` | `'lbfgs'` | Optimization algorithm: `'lbfgs'`, `'liblinear'`, `'newton-cg'`, `'sag'`, `'saga'` |
| `max_iter` | `100` | Maximum number of iterations for solver convergence |
| `class_weight` | `None` | Set to `'balanced'` to handle imbalanced classes |
| `multi_class` | `'auto'` | `'ovr'` (one-vs-rest) or `'multinomial'` for multiclass |

**Solver compatibility:**

| Solver | L1 | L2 | ElasticNet | Multinomial | Large datasets |
|---|---|---|---|---|---|
| `liblinear` | Yes | Yes | No | No (OvR only) | No |
| `lbfgs` | No | Yes | No | Yes | Moderate |
| `newton-cg` | No | Yes | No | Yes | Moderate |
| `sag` | No | Yes | No | Yes | Yes |
| `saga` | Yes | Yes | Yes | Yes | Yes |

**Interview point:** "Use `saga` solver for large datasets and when you need L1 or ElasticNet regularization. Use `lbfgs` (default) for most standard problems. Set `class_weight='balanced'` to handle imbalanced datasets."

---
## 12. Odds, Log-Odds, and Interpretation

Understanding odds and log-odds is critical for interpreting logistic regression coefficients.

**Probability to Odds:**

$$\text{Odds} = \frac{P(y=1)}{P(y=0)} = \frac{p}{1-p}$$

**Odds to Log-Odds (Logit):**

$$\text{logit}(p) = \log\left(\frac{p}{1-p}\right) = \mathbf{w}^T\mathbf{x}$$

**Coefficient interpretation:**
- A one-unit increase in $x_j$ changes the **log-odds** by $w_j$.
- Equivalently, it multiplies the **odds** by $e^{w_j}$.

**Example:** If $w_j = 0.7$, then a 1-unit increase in $x_j$ multiplies the odds by $e^{0.7} \approx 2.01$, meaning the odds roughly double.

**Real-world example:** In a loan default model, if the coefficient for "number of late payments" is 0.5, then each additional late payment multiplies the odds of default by $e^{0.5} \approx 1.65$ (65% increase in odds).

**Interview point:** "Logistic regression coefficients represent change in log-odds per unit change in the feature. To convert to odds ratio, exponentiate the coefficient. A positive coefficient increases the odds of the positive class; a negative coefficient decreases it."

---
## 13. Logistic Regression vs. Linear Regression -- Summary

| Aspect | Linear Regression | Logistic Regression |
|---|---|---|
| **Type** | Regression | Classification |
| **Output** | Continuous value $(-\infty, +\infty)$ | Probability $\in (0, 1)$ |
| **Function** | $\hat{y} = \mathbf{w}^T\mathbf{x}$ | $\hat{y} = \sigma(\mathbf{w}^T\mathbf{x})$ |
| **Cost function** | MSE (Mean Squared Error) | Log Loss (Binary Cross-Entropy) |
| **Decision boundary** | N/A | Linear in feature space |
| **Estimation** | OLS / Gradient Descent | MLE / Gradient Descent |
| **Assumptions** | Linearity, normality of errors, homoscedasticity | Linearity in log-odds, independence, no multicollinearity |

---
## 14. Interview Questions & Answers

### Q1. What is logistic regression? Is it a regression or classification algorithm?

Logistic regression is a **classification algorithm** despite its name. It models the probability that an input belongs to a particular class by applying the sigmoid function to a linear combination of features. The name comes from the logistic (sigmoid) function used, not from predicting continuous values.

---

### Q2. Why do we use the sigmoid function?

The sigmoid function maps any real-valued number to the range $(0, 1)$, making the output interpretable as a probability. It is differentiable everywhere, which allows gradient-based optimization. Its derivative $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ is efficiently computable.

---

### Q3. Why not use MSE as the cost function for logistic regression?

When MSE is used with the sigmoid function, the resulting cost surface is **non-convex** with multiple local minima. Gradient descent may converge to a suboptimal solution. Log loss (binary cross-entropy) produces a **convex** cost surface, guaranteeing convergence to the global minimum.

---

### Q4. What are the assumptions of logistic regression?

1. Binary or categorical dependent variable  
2. Independence of observations  
3. Little or no multicollinearity among predictors  
4. Linear relationship between features and log-odds  
5. Sufficiently large sample size  

It does NOT assume normally distributed features or homoscedasticity.

---

### Q5. How do you interpret coefficients in logistic regression?

Each coefficient $w_j$ represents the change in **log-odds** for a one-unit increase in feature $x_j$, holding all other features constant. The **odds ratio** $e^{w_j}$ gives the multiplicative change in odds. For example, $w_j = 0.5$ means each unit increase in $x_j$ multiplies the odds by $e^{0.5} \approx 1.65$.

---

### Q6. What is the decision boundary in logistic regression?

The decision boundary is the set of points where $\sigma(z) = 0.5$, which means $z = \mathbf{w}^T\mathbf{x} = 0$. It is always a **linear boundary** (hyperplane) in the original feature space. Non-linear boundaries require polynomial or interaction features.

---

### Q7. Can logistic regression handle non-linear relationships?

Not directly. Logistic regression creates a linear decision boundary. However, you can introduce non-linearity by:
- Adding polynomial features ($x_1^2$, $x_1 x_2$, etc.)
- Feature engineering (log transforms, binning)
- Using kernel methods

---

### Q8. How does logistic regression handle multiclass problems?

Two main approaches:
- **One-vs-Rest (OvR):** Train K separate binary classifiers, one for each class. Predict the class with the highest probability.
- **Multinomial (Softmax):** Generalize the sigmoid to the softmax function, directly outputting probabilities for all K classes that sum to 1.

---

### Q9. What happens when features are highly correlated (multicollinearity)?

Multicollinearity makes coefficient estimates **unstable** -- small changes in data cause large changes in coefficients. The model may still predict well overall, but individual coefficient interpretation becomes unreliable. Solutions: remove correlated features, use PCA, or apply L2 (Ridge) regularization.

---

### Q10. What is the relationship between logistic regression and Maximum Likelihood Estimation?

Logistic regression parameters are found by maximizing the **likelihood** of observed data. The log loss cost function is the **negative log-likelihood**. Minimizing log loss = maximizing the probability that the model correctly predicts the training labels.

---

### Q11. When should you use logistic regression over other classifiers?

Use logistic regression when:
- You need a **probabilistic output** (not just class labels)
- **Interpretability** matters (coefficients have clear meaning)
- The relationship between features and log-odds is approximately linear
- You have a **baseline model** requirement (it is fast and simple)
- You are working in **regulated industries** (finance, healthcare) where model explainability is required

---

### Q12. Real-world applications of logistic regression?

| Domain | Application |
|---|---|
| Healthcare | Disease diagnosis (malignant/benign), patient readmission prediction |
| Finance | Credit scoring, loan default prediction, fraud detection |
| Marketing | Customer churn prediction, click-through rate prediction |
| NLP | Spam detection, sentiment analysis (as baseline) |
| HR | Employee attrition prediction |

---
## 15. Quick Reference Summary

| Concept | Formula / Key Point |
|---|---|
| Sigmoid | $\sigma(z) = \frac{1}{1+e^{-z}}$ |
| Prediction | $\hat{y} = \sigma(\mathbf{w}^T\mathbf{x})$ |
| Log Loss | $J = -\frac{1}{m}\sum[y\log\hat{y} + (1-y)\log(1-\hat{y})]$ |
| Gradient | $\nabla J = \frac{1}{m}\mathbf{X}^T(\hat{\mathbf{y}} - \mathbf{y})$ |
| Decision Boundary | $\mathbf{w}^T\mathbf{x} = 0$ (at threshold 0.5) |
| Odds Ratio | $e^{w_j}$ = multiplicative change in odds per unit change in $x_j$ |
| Key Assumption | Linearity in log-odds, NOT in probability |
| Estimation Method | Maximum Likelihood Estimation (MLE) |

---
*Session 1 complete. Follow-up sessions will cover: regularization (L1/L2/ElasticNet), multiclass strategies, model evaluation metrics, and advanced diagnostics.*